# Classification: prepare once, predict and reopen

Default execution uses a synthetic HTTP fixture; it does not run an FM or measure quality. Live opt-in uses the qualified TabICLv2 classifier only, 128 training rows, four features and 32 query rows. No model fallback or hidden sampling. Credentials stay in local environment variables. Read README.md before live use.

In [ ]:
from support import notebook_client, prepare
import numpy as np
client = notebook_client()

Upload a synthetic table and wait for durable preparation. The returned predictor pins its model/configuration/seed. A local wait timeout does not cancel server work.

In [ ]:
columns, query, predictor = prepare(client)
assert predictor["task"] == "classification"
assert predictor["model_id"] == "tabicl-v2"
assert len(predictor["classes"]) == 2
pinned_version = predictor["model_version"]

Probabilities use the persisted class order. These are native model outputs, not conformal intervals or a calibration guarantee. Keep output row order.

In [ ]:
result = client.predict(predictor["id"], columns=columns, rows=query.tolist(), probabilities=True)
probabilities = np.asarray(result["probabilities"])
assert probabilities.shape == (32, 2)
assert np.isfinite(probabilities).all()
assert np.allclose(probabilities.sum(axis=1), 1, atol=1e-5)
assert result["classes"] == predictor["classes"]
assert result["model_version"] == pinned_version
assert len(result["predictions"]) == len(query)

Reopen by identity without uploading training data. Repeated predictions must preserve labels and the pinned model version. This checks reopen, not a forced worker cold restart.

In [ ]:
reopened = client.predictor(predictor["id"])
repeat = client.predict(reopened["id"], columns=columns, rows=query.tolist(), probabilities=True)
assert repeat["predictions"] == result["predictions"]
assert repeat["model_version"] == pinned_version
assert np.allclose(repeat["probabilities"], probabilities, atol=1e-5, rtol=1e-5)
client.close()

Regression, other selected models, full sklearn compatibility, causal discovery and batch workflows require their own released examples and qualification. This notebook does not satisfy those later gates.